# 06 — End-to-End Preparation for Machine Learning

## Task 2 — Sleep-profile multiclass classification

This notebook integrates the preparation decisions introduced in notebooks 01–05 into one reproducible workflow that starts from the raw tabular file and ends with model-ready TRAIN, VALIDATION, and TEST matrices.

The notebook **does not train a predictive model**. Its purpose is to freeze the preparation boundary that later model-training notebooks can reuse.

The central rule is:

```text
split first
fit data-dependent preparation on TRAIN only
transform VALIDATION and TEST without refitting
```

The workflow also exports a fitted preparation object and demonstrates that it can be reloaded and applied to a target-free held-out batch.


## Learning objectives

By the end of this notebook, you should be able to:

- connect deterministic preprocessing, feature engineering, splitting, scaling, and feature selection;
- distinguish operations that are fixed by a data contract from operations learned from data;
- construct a reusable scikit-learn preparation object;
- fit all data-dependent transformations using TRAIN only;
- transform VALIDATION and TEST with the frozen preparation;
- export model-ready split tables and reproducibility metadata;
- serialize both an unfitted preparation template and the fitted preparation artifact;
- reload the fitted artifact and prepare new target-free observations;
- understand why cross-validation later requires refitting the preparation workflow inside each training fold.


## 1. Workflow decision for this task

This multiclass demonstration uses a **stratified** split to preserve all four sleep-profile classes across TRAIN, VALIDATION, and TEST. Human-readable class labels are preserved in exports, while a separate `LabelEncoder` is fitted on TRAIN and serialized for algorithms that require numerical target codes.

The 70/15/15 split used here is a **teaching choice for the upcoming fixed-split training/validation/testing demonstrations**. It is not a universal recommendation. When data are scarce, a held-out TEST set plus cross-validation inside TRAIN may use the available observations more efficiently.


## 2. Imports and reproducibility

This is the first tabular-data notebook in the sequence where using a scikit-learn `Pipeline` is intentional. Earlier notebooks separated the techniques so that each operation could be inspected individually. Here the goal is different: preserve the fitted preparation as one reusable artifact.

The pipeline created below contains **preparation only**. No estimator is attached.


In [1]:
from pathlib import Path
import json
import re
import sys

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15
SELECTION_FRACTION = 0.60

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)


Python: 3.14.8
pandas: 3.0.6
scikit-learn: 1.9.1
joblib: 1.6.0


## 3. Locate the project and define outputs

Generated model-ready tables are written under the dataset directory. Reusable preparation artifacts and manifests are written under `results/`.

These generated outputs are products of executing the notebook; the notebook itself remains the reproducible source of the workflow.


In [2]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the repository root.")

PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_DIR = PROJECT_ROOT / "data" / "synthetic_sleep_profiles"
RAW_DIR = DATASET_DIR / "raw_data"
RAW_DATA_PATH = RAW_DIR / "synthetic_sleep_profiles_1200.csv"
METADATA_PATH = RAW_DIR / "metadata.json"

ML_READY_DIR = DATASET_DIR / "ml_ready"
RESULTS_DIR = PROJECT_ROOT / "results" / "synthetic_sleep_profiles" / "ml_preparation"
ARTIFACTS_DIR = RESULTS_DIR / "artifacts"
TABLES_DIR = RESULTS_DIR / "tables"

for directory in [ML_READY_DIR, ARTIFACTS_DIR, TABLES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA_PATH.relative_to(PROJECT_ROOT))
print("ML-ready output:", ML_READY_DIR.relative_to(PROJECT_ROOT))
print("Preparation artifacts:", ARTIFACTS_DIR.relative_to(PROJECT_ROOT))


Project root: /home/david/Desktop/depto_computacion/ml_2026/demos_machine_learning_2026
Raw data: data/synthetic_sleep_profiles/raw_data/synthetic_sleep_profiles_1200.csv
ML-ready output: data/synthetic_sleep_profiles/ml_ready
Preparation artifacts: results/synthetic_sleep_profiles/ml_preparation/artifacts


## 4. Load the raw table and metadata

The end-to-end workflow starts from the raw table instead of assuming that an earlier notebook has already been executed. The metadata acts as the input contract for column roles.


In [3]:
with METADATA_PATH.open("r", encoding="utf-8") as file:
    metadata = json.load(file)

raw_df = pd.read_csv(RAW_DATA_PATH)

print("Raw shape:", raw_df.shape)
display(raw_df.head())
print("Metadata target:", metadata["label"])
print("Metadata identifier:", metadata["col_id"])


Raw shape: (1200, 23)


,participant_id,age,sex_female,height_cm,weight_kg,bmi,sleep_duration_hours,sleep_efficiency_pct,sleep_latency_min,night_awakenings,...,screen_time_evening_min,stress_score_0_10,social_jetlag_hours,bedtime_variability_min,shift_worker,regular_exercise,evening_alcohol,loud_snoring,anxiety_history,sleep_profile
0,SP0001,32,0,167.8,81.2,28.8,6.93,72.8,34.1,2,...,78,5.0,1.59,98,1,0,0,0,0,circadian_disruption
1,SP0002,49,0,177.2,72.5,23.1,6.85,88.8,27.9,1,...,158,6.1,2.44,120,1,1,0,0,0,circadian_disruption
2,SP0003,18,0,161.7,54.1,20.7,7.68,78.5,19.6,3,...,113,0.5,1.41,42,0,1,0,0,0,healthy_sleep
3,SP0004,67,0,188.9,96.8,27.1,7.79,89.5,4.2,3,...,16,3.1,0.54,13,0,1,1,0,0,healthy_sleep
4,SP0005,55,1,155.4,53.0,21.9,6.44,75.6,49.6,4,...,71,6.9,1.12,48,0,1,0,0,0,insomnia


Metadata target: sleep_profile
Metadata identifier: participant_id


## 5. Deterministic structural preprocessing

The following function performs only contract-based operations:

- normalize column names;
- validate required columns;
- remove exact duplicate rows;
- require unique, non-missing identifiers;
- coerce declared predictors to numerical dtype;
- validate non-missing binary indicators as 0/1;
- validate the target when it is present.

Predictor missing values are **not imputed here**. Imputation is data-dependent and therefore belongs in the fitted training-only preparation object.


In [4]:
def normalize_column_name(name: str) -> str:
    name = str(name).strip().lower()
    name = re.sub(r"[^0-9a-zA-Z]+", "_", name)
    return re.sub(r"_+", "_", name).strip("_")


def normalized_roles(metadata: dict) -> dict:
    return {
        "id": normalize_column_name(metadata["col_id"]),
        "target": normalize_column_name(metadata["label"]),
        "numerical": [normalize_column_name(c) for c in metadata["numerical_column"]],
        "categorical": [normalize_column_name(c) for c in metadata["categorical_column"]],
    }


ROLES = normalized_roles(metadata)
ID_COLUMN = ROLES["id"]
TARGET_COLUMN = ROLES["target"]
ORIGINAL_NUMERICAL = ROLES["numerical"]
CATEGORICAL_COLUMNS = ROLES["categorical"]


def clean_raw_table(frame: pd.DataFrame, require_target: bool = True) -> pd.DataFrame:
    out = frame.copy()
    out.columns = [normalize_column_name(c) for c in out.columns]

    required = [ID_COLUMN, *ORIGINAL_NUMERICAL, *CATEGORICAL_COLUMNS]
    if require_target:
        required.append(TARGET_COLUMN)

    missing_columns = sorted(set(required) - set(out.columns))
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    out = out.drop_duplicates().copy()

    if out[ID_COLUMN].isna().any():
        raise ValueError("Identifier contains missing values.")
    if out[ID_COLUMN].duplicated().any():
        raise ValueError("Identifier must be unique after exact-duplicate removal.")

    for column in [*ORIGINAL_NUMERICAL, *CATEGORICAL_COLUMNS]:
        out[column] = pd.to_numeric(out[column], errors="raise")

    for column in CATEGORICAL_COLUMNS:
        observed = set(out[column].dropna().unique().tolist())
        if not observed.issubset({0, 1}):
            raise ValueError(f"{column} contains values outside {0, 1}: {observed}")

    if TARGET_COLUMN in out.columns:
        if False:
            out[TARGET_COLUMN] = pd.to_numeric(out[TARGET_COLUMN], errors="raise")
        else:
            out[TARGET_COLUMN] = out[TARGET_COLUMN].astype("string").str.strip()

        if require_target and out[TARGET_COLUMN].isna().any():
            raise ValueError("Training/evaluation data cannot contain missing target values.")

    return out


clean_df = clean_raw_table(raw_df, require_target=True)
print("Clean shape:", clean_df.shape)
print("Exact duplicates after cleaning:", int(clean_df.duplicated().sum()))
print("Duplicated identifiers:", int(clean_df[ID_COLUMN].duplicated().sum()))
print("Predictor missing values retained for fitted imputation:",
      int(clean_df[ORIGINAL_NUMERICAL + CATEGORICAL_COLUMNS].isna().sum().sum()))


Clean shape: (1200, 23)
Exact duplicates after cleaning: 0
Duplicated identifiers: 0
Predictor missing values retained for fitted imputation: 0


## 6. Deterministic feature engineering

The feature recipe is fixed before the split and does not estimate statistics from the observed sample. The same function will therefore be applied to TRAIN, VALIDATION, TEST, and future external batches.

If this workflow were promoted from teaching material to a production system, this deterministic function would ideally live in tested source code rather than only inside a notebook.


In [5]:
def engineer_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()

    out["sleep_duration_deviation_8h"] = (out["sleep_duration_hours"] - 8.0).abs()
    out["sleep_efficiency_deficit_pct"] = 100.0 - out["sleep_efficiency_pct"]
    out["sleep_fragmentation_per_hour"] = (
        out["night_awakenings"] / out["sleep_duration_hours"].clip(lower=0.5)
    )
    out["circadian_disruption_index"] = (
        out["social_jetlag_hours"] + out["bedtime_variability_min"] / 60.0
    )
    out["behavioral_risk_count"] = (
        out[["shift_worker", "evening_alcohol", "anxiety_history"]].sum(axis=1, min_count=1)
        + (1.0 - out["regular_exercise"])
    )
    out = out.replace([np.inf, -np.inf], np.nan)
    return out


In [6]:
ENGINEERED_NUMERICAL = ['sleep_duration_deviation_8h', 'sleep_efficiency_deficit_pct', 'sleep_fragmentation_per_hour', 'circadian_disruption_index', 'behavioral_risk_count']
NUMERICAL_FEATURES = ORIGINAL_NUMERICAL + ENGINEERED_NUMERICAL
FEATURE_COLUMNS = NUMERICAL_FEATURES + CATEGORICAL_COLUMNS

preview_eng = engineer_features(clean_df.head(5))
assert set(FEATURE_COLUMNS).issubset(preview_eng.columns)

feature_roles = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "role": ["numerical"] * len(NUMERICAL_FEATURES) + ["binary_categorical"] * len(CATEGORICAL_COLUMNS),
    "origin": (
        ["original"] * len(ORIGINAL_NUMERICAL)
        + ["engineered"] * len(ENGINEERED_NUMERICAL)
        + ["original"] * len(CATEGORICAL_COLUMNS)
    ),
})
display(feature_roles)
print("Candidate predictors:", len(FEATURE_COLUMNS))


,feature,role,origin
0,age,numerical,original
1,height_cm,numerical,original
2,weight_kg,numerical,original
3,bmi,numerical,original
4,sleep_duration_hours,numerical,original
5,sleep_efficiency_pct,numerical,original
6,sleep_latency_min,numerical,original
7,night_awakenings,numerical,original
8,resting_heart_rate_bpm,numerical,original
9,daily_steps,numerical,original


Candidate predictors: 26


## 7. Create TRAIN, VALIDATION, and TEST before fitting preparation

All fitted preprocessing decisions must occur **after** this cell. The identifier is used only to audit the partition and is never passed as a predictor.


In [7]:
train_df, temporary_df = train_test_split(
    clean_df,
    test_size=(VALIDATION_FRACTION + TEST_FRACTION),
    random_state=RANDOM_STATE,
    stratify=clean_df[TARGET_COLUMN],
)
validation_df, test_df = train_test_split(
    temporary_df,
    test_size=TEST_FRACTION / (VALIDATION_FRACTION + TEST_FRACTION),
    random_state=RANDOM_STATE,
    stratify=temporary_df[TARGET_COLUMN],
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)


In [8]:
assert set(train_df[ID_COLUMN]).isdisjoint(validation_df[ID_COLUMN])
assert set(train_df[ID_COLUMN]).isdisjoint(test_df[ID_COLUMN])
assert set(validation_df[ID_COLUMN]).isdisjoint(test_df[ID_COLUMN])
assert len(train_df) + len(validation_df) + len(test_df) == len(clean_df)

split_summary = pd.DataFrame({
    "split": ["TRAIN", "VALIDATION", "TEST"],
    "n": [len(train_df), len(validation_df), len(test_df)],
    "fraction": [len(train_df)/len(clean_df), len(validation_df)/len(clean_df), len(test_df)/len(clean_df)],
})
display(split_summary)


,split,n,fraction
0,TRAIN,840,0.70
1,VALIDATION,180,0.15
2,TEST,180,0.15


## 8. Apply the deterministic feature recipe to each partition

This step uses the same formula definitions everywhere. It does not learn from VALIDATION or TEST.


In [9]:
train_eng = engineer_features(train_df)
validation_eng = engineer_features(validation_df)
test_eng = engineer_features(test_df)

X_train = train_eng[FEATURE_COLUMNS].copy()
X_validation = validation_eng[FEATURE_COLUMNS].copy()
X_test = test_eng[FEATURE_COLUMNS].copy()

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_test:", X_test.shape)


X_train: (840, 26)
X_validation: (180, 26)
X_test: (180, 26)


## 9. Prepare the target

Predictor preparation and target representation are separate concerns. The target transformation used below is task-specific and is saved when future interpretation requires an inverse mapping.


In [10]:
target_encoder = LabelEncoder()
y_train = target_encoder.fit_transform(train_df[TARGET_COLUMN].astype(str))
y_validation = target_encoder.transform(validation_df[TARGET_COLUMN].astype(str))
y_test = target_encoder.transform(test_df[TARGET_COLUMN].astype(str))
y_train_for_preparation = y_train

expected_classes = set(metadata.get("classes", []))
observed_classes = set(target_encoder.classes_.tolist())
if expected_classes and expected_classes != observed_classes:
    raise ValueError(f"Target classes differ from metadata: {observed_classes} vs {expected_classes}")

class_mapping = pd.DataFrame({
    "encoded": np.arange(len(target_encoder.classes_)),
    "class_name": target_encoder.classes_,
})
display(class_mapping)


,encoded,class_name
0,0,circadian_disruption
1,1,healthy_sleep
2,2,insomnia
3,3,sleep_apnea


## 10. Define an unfitted reusable preparation template

The template contains only scikit-learn objects that can be serialized safely without depending on notebook-local transformation functions.

The deterministic feature recipe remains explicit in this notebook. The fitted pipeline begins at the candidate-feature table and performs:

```text
numerical predictors
    median imputation
    StandardScaler

binary predictors
    most-frequent imputation

all prepared predictors
    supervised SelectKBest
```

The selector retains 60% of the candidate predictors for this teaching example. That fraction is not presented as an optimum; later modelling can compare different preparation choices.


In [11]:
SELECTION_K = max(1, int(np.ceil(SELECTION_FRACTION * len(FEATURE_COLUMNS))))

numerical_preparation = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

binary_preparation = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

column_preparation = ColumnTransformer(
    transformers=[
        ("numerical", numerical_preparation, NUMERICAL_FEATURES),
        ("binary", binary_preparation, CATEGORICAL_COLUMNS),
    ],
    remainder="drop",
    sparse_threshold=0.0,
    verbose_feature_names_out=False,
)

preparation_template = Pipeline([
    ("columns", column_preparation),
    ("selector", SelectKBest(score_func=f_classif, k=SELECTION_K)),
])

print(preparation_template)
print(f"Selector retains {SELECTION_K} of {len(FEATURE_COLUMNS)} candidate predictors.")


Pipeline(steps=[('columns',
                 ColumnTransformer(sparse_threshold=0.0,
                                   transformers=[('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'height_cm',
                                                   'weight_kg', 'bmi',
                                                   'sleep_duration_hours',
                                                   'sleep_efficiency_pct',
                                                   'sleep_latency_min',
                                                   'night_awakenings',
                                                   'resting_heart_rate_b

## 11. Save the unfitted template

The unfitted template is useful later if a modelling notebook performs cross-validation. It can be cloned or combined with an estimator so the complete preparation is refitted inside every training fold.


In [12]:
TEMPLATE_PATH = ARTIFACTS_DIR / "preparation_pipeline_template.joblib"
joblib.dump(clone(preparation_template), TEMPLATE_PATH)
print("Saved:", TEMPLATE_PATH.relative_to(PROJECT_ROOT))


Saved: results/synthetic_sleep_profiles/ml_preparation/artifacts/preparation_pipeline_template.joblib


## 12. Fit preparation using TRAIN only

This is the fitting boundary. Imputation statistics, scaling parameters, and supervised feature-selection scores are learned here and nowhere else.


In [13]:
preparation_fitted = clone(preparation_template)
preparation_fitted.fit(X_train, y_train_for_preparation)

X_train_ready = preparation_fitted.transform(X_train)
X_validation_ready = preparation_fitted.transform(X_validation)
X_test_ready = preparation_fitted.transform(X_test)

pre_feature_names = preparation_fitted.named_steps["columns"].get_feature_names_out()
selection_mask = preparation_fitted.named_steps["selector"].get_support()
SELECTED_FEATURES = pre_feature_names[selection_mask].tolist()

assert X_train_ready.shape[1] == len(SELECTED_FEATURES)
assert X_validation_ready.shape[1] == len(SELECTED_FEATURES)
assert X_test_ready.shape[1] == len(SELECTED_FEATURES)
assert np.isfinite(X_train_ready).all()
assert np.isfinite(X_validation_ready).all()
assert np.isfinite(X_test_ready).all()

print("Selected predictors:", len(SELECTED_FEATURES))
display(pd.DataFrame({"selected_feature": SELECTED_FEATURES}))


Selected predictors: 16


,selected_feature
0,age
1,bmi
2,sleep_efficiency_pct
3,sleep_latency_min
4,night_awakenings
5,caffeine_mg_day
6,screen_time_evening_min
7,stress_score_0_10
8,social_jetlag_hours
9,bedtime_variability_min


## 13. Inspect training-derived fitted parameters

This cell is an audit, not another fitting step. The values shown below were learned exclusively from TRAIN.


In [14]:
num_pipeline_fitted = preparation_fitted.named_steps["columns"].named_transformers_["numerical"]
imputer_statistics = pd.Series(
    num_pipeline_fitted.named_steps["imputer"].statistics_,
    index=NUMERICAL_FEATURES,
    name="training_median",
)
scaler_means = pd.Series(
    num_pipeline_fitted.named_steps["scaler"].mean_,
    index=NUMERICAL_FEATURES,
    name="training_mean_after_imputation",
)
scaler_scales = pd.Series(
    num_pipeline_fitted.named_steps["scaler"].scale_,
    index=NUMERICAL_FEATURES,
    name="training_scale",
)

fitted_numeric_parameters = pd.concat(
    [imputer_statistics, scaler_means, scaler_scales], axis=1
)
display(fitted_numeric_parameters.head(12))


,training_median,training_mean_after_imputation,training_scale
age,45.50,44.822619,15.259368
height_cm,169.85,170.088333,9.385741
weight_kg,76.35,77.458929,16.557751
bmi,26.60,26.700714,4.850542
sleep_duration_hours,6.91,6.918298,0.801560
sleep_efficiency_pct,81.25,81.063690,5.318127
sleep_latency_min,31.45,31.886667,10.944105
night_awakenings,1.00,1.710714,1.400129
resting_heart_rate_bpm,70.00,69.621429,7.327997
daily_steps,8325.50,8397.019048,2315.854497


## 14. Export model-ready fixed splits

The transformed tables are convenient for the upcoming **fixed train/validation/test modelling demonstrations**. They include the identifier for traceability but the identifier is not part of the prepared predictor matrix.

For cross-validation, prefer loading the unfitted preparation template and refitting it inside each fold instead of cross-validating directly on a matrix produced by a selector already fitted to the full TRAIN partition.


In [15]:
def attach_target(frame, split: str):
    out = frame.copy()
    original = {
        "train": train_df[TARGET_COLUMN].astype(str).to_numpy(),
        "validation": validation_df[TARGET_COLUMN].astype(str).to_numpy(),
        "test": test_df[TARGET_COLUMN].astype(str).to_numpy(),
    }[split]
    encoded = {
        "train": y_train,
        "validation": y_validation,
        "test": y_test,
    }[split]
    out[TARGET_COLUMN] = original
    out[f"{TARGET_COLUMN}_encoded"] = encoded
    return out


In [16]:
def ready_frame(ids, matrix, selected_features):
    out = pd.DataFrame(matrix, columns=selected_features)
    out.insert(0, ID_COLUMN, ids.reset_index(drop=True))
    return out

train_ready_df = ready_frame(train_df[ID_COLUMN], X_train_ready, SELECTED_FEATURES)
validation_ready_df = ready_frame(validation_df[ID_COLUMN], X_validation_ready, SELECTED_FEATURES)
test_ready_df = ready_frame(test_df[ID_COLUMN], X_test_ready, SELECTED_FEATURES)

train_ready_df = attach_target(train_ready_df, split="train")
validation_ready_df = attach_target(validation_ready_df, split="validation")
test_ready_df = attach_target(test_ready_df, split="test")

TRAIN_READY_PATH = ML_READY_DIR / "train_ready.csv"
VALIDATION_READY_PATH = ML_READY_DIR / "validation_ready.csv"
TEST_READY_PATH = ML_READY_DIR / "test_ready.csv"

train_ready_df.to_csv(TRAIN_READY_PATH, index=False)
validation_ready_df.to_csv(VALIDATION_READY_PATH, index=False)
test_ready_df.to_csv(TEST_READY_PATH, index=False)

print("Exported model-ready tables:")
for path in [TRAIN_READY_PATH, VALIDATION_READY_PATH, TEST_READY_PATH]:
    print(" -", path.relative_to(PROJECT_ROOT))

display(train_ready_df.head())


Exported model-ready tables:
 - data/synthetic_sleep_profiles/ml_ready/train_ready.csv
 - data/synthetic_sleep_profiles/ml_ready/validation_ready.csv
 - data/synthetic_sleep_profiles/ml_ready/test_ready.csv


,participant_id,age,bmi,sleep_efficiency_pct,sleep_latency_min,night_awakenings,caffeine_mg_day,screen_time_evening_min,stress_score_0_10,social_jetlag_hours,bedtime_variability_min,sleep_efficiency_deficit_pct,sleep_fragmentation_per_hour,circadian_disruption_index,behavioral_risk_count,shift_worker,anxiety_history,sleep_profile,sleep_profile_encoded
0,SP0545,1.846563,-1.092809,0.326489,-0.346001,-0.507606,-1.225767,2.045786,0.237270,-0.526751,0.206965,-0.326489,-0.323459,-0.276046,0.062428,0.0,1.0,insomnia,2
1,SP1118,1.060161,1.793467,-0.275227,-0.346001,0.920833,1.449385,0.056922,-0.062139,0.700902,-0.684237,0.275227,0.935784,0.191731,0.062428,0.0,0.0,sleep_apnea,3
2,SP0269,0.208225,-0.165077,0.439311,-0.364275,0.920833,0.829658,-0.332649,1.684414,-0.600410,-0.552208,-0.439311,0.804447,-0.653102,0.062428,0.0,1.0,insomnia,2
3,SP0804,0.666960,-0.701100,-1.403443,2.047982,0.920833,-0.141246,0.508005,1.385005,1.191963,1.725310,1.403443,1.386147,1.566711,0.062428,1.0,0.0,insomnia,2
4,SP1055,-1.757781,-0.494937,-0.576084,1.709901,-0.507606,0.499138,-0.804235,0.337073,-1.570255,-0.123110,0.576084,-0.481339,-1.140724,0.062428,0.0,0.0,healthy_sleep,1


## 15. Export split assignments

A random seed helps reproduce a split, but an explicit assignment table is a stronger record of which observations were used for each role.


In [17]:
split_assignments = pd.concat([
    train_df[[ID_COLUMN, TARGET_COLUMN]].assign(split="TRAIN"),
    validation_df[[ID_COLUMN, TARGET_COLUMN]].assign(split="VALIDATION"),
    test_df[[ID_COLUMN, TARGET_COLUMN]].assign(split="TEST"),
], ignore_index=True)

SPLIT_ASSIGNMENTS_PATH = TABLES_DIR / "split_assignments.csv"
split_assignments.to_csv(SPLIT_ASSIGNMENTS_PATH, index=False)
display(split_assignments.head())


,participant_id,sleep_profile,split
0,SP0545,insomnia,TRAIN
1,SP1118,sleep_apnea,TRAIN
2,SP0269,insomnia,TRAIN
3,SP0804,insomnia,TRAIN
4,SP1055,healthy_sleep,TRAIN


## 16. Save the fitted preparation artifact

This fitted object is the frozen predictor preparation learned from TRAIN. It can transform VALIDATION, TEST, and later external observations, provided the deterministic feature recipe and input schema are respected.


In [18]:
FITTED_PIPELINE_PATH = ARTIFACTS_DIR / "preparation_pipeline_fitted.joblib"
joblib.dump(preparation_fitted, FITTED_PIPELINE_PATH)
print("Saved:", FITTED_PIPELINE_PATH.relative_to(PROJECT_ROOT))


Saved: results/synthetic_sleep_profiles/ml_preparation/artifacts/preparation_pipeline_fitted.joblib


## 17. Save the target encoder

The predictor preparation object and target representation are kept as separate artifacts. The target encoder is needed later to convert numeric model outputs back to human-readable class names.


In [19]:
TARGET_ENCODER_PATH = ARTIFACTS_DIR / "target_label_encoder.joblib"
joblib.dump(target_encoder, TARGET_ENCODER_PATH)
print("Saved:", TARGET_ENCODER_PATH.relative_to(PROJECT_ROOT))
print("Inverse mapping check:", target_encoder.inverse_transform(np.arange(len(target_encoder.classes_))).tolist())


Saved: results/synthetic_sleep_profiles/ml_preparation/artifacts/target_label_encoder.joblib
Inverse mapping check: ['circadian_disruption', 'healthy_sleep', 'insomnia', 'sleep_apnea']


## 18. Export the preparation manifest

The serialized object is useful for computation; the manifest is useful for auditability. It records the choices needed to understand what the object expects and what it produces.


In [20]:
manifest = {
    "dataset": "synthetic_sleep_profiles",
    "task_type": "multiclass_classification",
    "source": str(RAW_DATA_PATH.relative_to(PROJECT_ROOT)),
    "identifier": ID_COLUMN,
    "target": TARGET_COLUMN,
    "random_state": RANDOM_STATE,
    "split_fractions": {"train": TRAIN_FRACTION, "validation": VALIDATION_FRACTION, "test": TEST_FRACTION},
    "split_sizes": {"train": len(train_df), "validation": len(validation_df), "test": len(test_df)},
    "original_numerical_features": ORIGINAL_NUMERICAL,
    "engineered_numerical_features": ENGINEERED_NUMERICAL,
    "binary_categorical_features": CATEGORICAL_COLUMNS,
    "candidate_features": FEATURE_COLUMNS,
    "selection_method": "SelectKBest(f_classif)",
    "selection_fraction": SELECTION_FRACTION,
    "selection_k": SELECTION_K,
    "selected_features": SELECTED_FEATURES,
    "prepared_dimensions": {
        "train": list(X_train_ready.shape),
        "validation": list(X_validation_ready.shape),
        "test": list(X_test_ready.shape),
    },
    "artifacts": {
        "unfitted_preparation_template": str(TEMPLATE_PATH.relative_to(PROJECT_ROOT)),
        "fitted_preparation": str(FITTED_PIPELINE_PATH.relative_to(PROJECT_ROOT)),
        "split_assignments": str(SPLIT_ASSIGNMENTS_PATH.relative_to(PROJECT_ROOT)),
    },
    "software": {
        "python": sys.version.split()[0],
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },
    "fixed_split_note": (
        "The fitted preparation artifact is valid for the fixed TRAIN/VALIDATION/TEST workflow "
        "and for later target-free external observations transformed without refitting."
    ),
    "cross_validation_warning": (
        "If later model development uses cross-validation, load/rebuild the UNFITTED preparation "
        "template and fit it inside each training fold together with the estimator. Do not perform "
        "cross-validation on features selected once using the complete TRAIN partition."
    ),
}

manifest["target_classes"] = target_encoder.classes_.tolist()
manifest["artifacts"]["target_encoder"] = str(TARGET_ENCODER_PATH.relative_to(PROJECT_ROOT))

MANIFEST_PATH = RESULTS_DIR / "preparation_manifest.json"
with MANIFEST_PATH.open("w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2)

SELECTED_FEATURES_PATH = RESULTS_DIR / "selected_feature_names.json"
with SELECTED_FEATURES_PATH.open("w", encoding="utf-8") as file:
    json.dump(SELECTED_FEATURES, file, indent=2)

print("Saved:", MANIFEST_PATH.relative_to(PROJECT_ROOT))
print("Saved:", SELECTED_FEATURES_PATH.relative_to(PROJECT_ROOT))


Saved: results/synthetic_sleep_profiles/ml_preparation/preparation_manifest.json
Saved: results/synthetic_sleep_profiles/ml_preparation/selected_feature_names.json


## 19. Reload the fitted artifact and prepare target-free external data

A reusable preparation workflow should work when the target is unavailable. To test that boundary, the next cells reconstruct a small batch from held-out raw observations, remove the target, run the same deterministic cleaning and feature recipe, reload the preparation object from disk, and call **`transform()` only**.

No parameter is refitted on this batch.


In [21]:
external_ids = test_df[ID_COLUMN].head(5).tolist()
raw_id_column = metadata["col_id"]
raw_target_column = metadata["label"]

external_raw = (
    raw_df.set_index(raw_id_column)
    .loc[external_ids]
    .reset_index()
    .drop(columns=[raw_target_column])
)

external_clean = clean_raw_table(external_raw, require_target=False)
external_eng = engineer_features(external_clean)
external_X = external_eng[FEATURE_COLUMNS]

loaded_preparation = joblib.load(FITTED_PIPELINE_PATH)
external_ready = loaded_preparation.transform(external_X)

np.testing.assert_allclose(
    external_ready,
    X_test_ready[: len(external_ids)],
    rtol=1e-10,
    atol=1e-10,
)

external_ready_df = pd.DataFrame(external_ready, columns=SELECTED_FEATURES)
external_ready_df.insert(0, ID_COLUMN, external_ids)

display(external_ready_df)
print("Reload + external transform check: PASS")


,participant_id,age,bmi,sleep_efficiency_pct,sleep_latency_min,night_awakenings,caffeine_mg_day,screen_time_evening_min,stress_score_0_10,social_jetlag_hours,bedtime_variability_min,sleep_efficiency_deficit_pct,sleep_fragmentation_per_hour,circadian_disruption_index,behavioral_risk_count,shift_worker,anxiety_history
0,SP0162,2.436364,-0.804181,-0.801728,0.576871,1.635053,0.798672,-1.009272,0.137467,-1.717573,-0.882282,0.801728,1.120610,-1.568810,1.281959,0.0,0.0
1,SP0549,-1.430113,0.700805,0.815383,0.183965,-1.221826,0.602426,0.836065,-0.760761,0.037970,-1.476417,-0.815383,-1.204404,-0.607742,0.062428,0.0,0.0
2,SP0617,1.256761,0.968817,-2.381231,1.956609,1.635053,0.220261,0.323471,1.385005,1.363834,-0.816267,2.381231,1.727599,0.594302,0.062428,0.0,1.0
3,SP0631,0.273758,1.010049,0.984615,-0.866829,-0.507606,-0.151575,0.343975,1.035694,0.909603,0.471025,-0.984615,-0.570170,0.832443,0.062428,0.0,0.0
4,SP0067,-0.578177,-1.360820,-0.876942,1.554566,-0.507606,1.759248,0.036419,-0.611056,-0.612686,0.206965,0.876942,-0.451872,-0.335581,-1.157102,0.0,0.0


Reload + external transform check: PASS


## 20. Final validation

Before handing these outputs to a modelling notebook, verify the preparation contract:

- partitions are disjoint;
- preparation was fitted once using TRAIN only;
- VALIDATION and TEST were transformed without refitting;
- the prepared matrices contain no missing or infinite values;
- identifiers are not predictors;
- selected feature names are frozen;
- the fitted object survives serialization and reload;
- a target-free held-out batch produces the same prepared representation;
- cross-validation later will use the unfitted template rather than the already selected TRAIN matrix.


In [22]:
checks = {
    "train_validation_disjoint": set(train_df[ID_COLUMN]).isdisjoint(validation_df[ID_COLUMN]),
    "train_test_disjoint": set(train_df[ID_COLUMN]).isdisjoint(test_df[ID_COLUMN]),
    "validation_test_disjoint": set(validation_df[ID_COLUMN]).isdisjoint(test_df[ID_COLUMN]),
    "train_ready_finite": bool(np.isfinite(X_train_ready).all()),
    "validation_ready_finite": bool(np.isfinite(X_validation_ready).all()),
    "test_ready_finite": bool(np.isfinite(X_test_ready).all()),
    "identifier_excluded": ID_COLUMN not in SELECTED_FEATURES,
    "target_excluded": TARGET_COLUMN not in SELECTED_FEATURES,
    "serialized_pipeline_exists": FITTED_PIPELINE_PATH.exists(),
    "template_exists": TEMPLATE_PATH.exists(),
    "manifest_exists": MANIFEST_PATH.exists(),
}

validation_table = pd.DataFrame({"check": checks.keys(), "passed": checks.values()})
display(validation_table)

if not all(checks.values()):
    raise AssertionError("At least one final preparation check failed.")

print("FINAL PREPARATION VALIDATION: PASS")


,check,passed
0,train_validation_disjoint,True
1,train_test_disjoint,True
2,validation_test_disjoint,True
3,train_ready_finite,True
4,validation_ready_finite,True
5,test_ready_finite,True
6,identifier_excluded,True
7,target_excluded,True
8,serialized_pipeline_exists,True
9,template_exists,True


FINAL PREPARATION VALIDATION: PASS


# Final interpretation

This notebook has frozen a **data-preparation boundary**, not a predictive model.

The next modelling notebooks can now take two complementary routes:

### Fixed split teaching workflow

Use the exported `train_ready.csv`, `validation_ready.csv`, and `test_ready.csv` files to demonstrate model fitting, model selection, and final testing with a simple fixed partition.

### Cross-validation workflow

Load the **unfitted** preparation template and combine it with an estimator. The preparation and estimator should then be fitted together inside every training fold.

For future external prediction, the eventual trained model must receive observations that pass through the same deterministic feature recipe and the same fitted preparation logic used during model development.
